In [1]:
!pip install ultralytics torchtoolbox bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 20.3 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
from ultralytics import YOLO
from torch.nn import MultiheadAttention
from torchvision import transforms
from PIL import Image
import random
import os
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# ============================================================================
# CONSTANTS
# ============================================================================
CHARACTERS = '0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ'
SOS_TOKEN = 36
EOS_TOKEN = 37
PAD_TOKEN = len(CHARACTERS) + 2  # = 38
NUM_CLASSES = len(CHARACTERS) + 3  # = 39 (0-35 chars, 36 SOS, 37 EOS, 38 PAD)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def index_to_char(indices, include_special_tokens=False):
    """Chuyển list index thành chuỗi ký tự, dừng lại tại EOS."""
    result = []
    for i in indices:
        i = i.item() if torch.is_tensor(i) else i
        if i == SOS_TOKEN:
            if include_special_tokens:
                result.append('[SOS]')
        elif i == EOS_TOKEN:
            if include_special_tokens:
                result.append('[EOS]')
            break
        elif i == PAD_TOKEN:
            break
        elif 0 <= i < len(CHARACTERS):
            result.append(CHARACTERS[i])
        else:
            if include_special_tokens:
                result.append(f'[UNK_{i}]')
    return ''.join(result)


def char_to_indices(text):
    """Chuyển chuỗi text thành tensor indices với SOS ở đầu, EOS ở cuối."""
    indices = [SOS_TOKEN]
    for c in text:
        if c in CHARACTERS:
            indices.append(CHARACTERS.index(c))
        else:
            indices.append(0)  # Map unknown char to '0'
    indices.append(EOS_TOKEN)
    return torch.tensor(indices, dtype=torch.long)


# Dùng chung cho cả train và val, tránh lặp code
def extract_pred_and_true(pred_indices, true_indices):
    # Pred: cắt tại EOS hoặc PAD (whichever comes first)
    pred_content = []
    for idx in pred_indices:
        if idx == EOS_TOKEN or idx == PAD_TOKEN:
            break
        pred_content.append(idx)
    
    # True: lọc bỏ EOS và PAD, chỉ giữ ký tự thực
    true_content = [x for x in true_indices if x not in (EOS_TOKEN, PAD_TOKEN)]
    
    return pred_content, true_content


def compute_crr(pred_content, true_content):
    total = max(len(pred_content), len(true_content))
    if total == 0:
        return 0, 0
    
    correct = sum(
        p == t for p, t in zip(pred_content, true_content)
    )
    return correct, total


# ============================================================================
# YOLO BACKBONE
# ============================================================================
class YoloBackbone(nn.Module):
    def __init__(self, model_path, shallow_layer=6, deep_layer=13):
        super().__init__()
        _temp = YOLO(model_path)
        self.yolo_detection_model = _temp.model
        self.yolo_detection_model.to(DEVICE)
        self.shallow_layer = shallow_layer
        self.deep_layer = deep_layer

        for param in self.yolo_detection_model.parameters():
            param.requires_grad = True

        self._fmaps = {}
        self._hooks = []
        self._register_hooks()

    def _make_hook(self, name):
        def hook_fn(module, inp, out):
            if isinstance(out, torch.Tensor):
                self._fmaps[name] = out
            elif isinstance(out, (list, tuple)):
                for item in out:
                    if isinstance(item, torch.Tensor):
                        self._fmaps[name] = item
                        break
        return hook_fn

    def _register_hooks(self):
        for idx in [self.shallow_layer, self.deep_layer]:
            layer = self.yolo_detection_model.model[idx]
            h = layer.register_forward_hook(self._make_hook(idx))
            self._hooks.append(h)

    def forward(self, x):
        self._fmaps.clear()
        _ = self.yolo_detection_model(x)
        shallow = self._fmaps[self.shallow_layer]  # high-res, low-semantic
        deep = self._fmaps[self.deep_layer]         # low-res, high-semantic
        return shallow, deep

class LightFusion(nn.Module):
    """Fuse 2 scale features. Rất nhẹ — chỉ 1 conv + 1 addition."""
    def __init__(self, shallow_ch, deep_ch, out_ch):
        super().__init__()
        # Đưa shallow về cùng channels với deep
        self.shallow_proj = nn.Sequential(
            nn.Conv2d(shallow_ch, out_ch, 1),  # 1x1 conv, rất nhanh
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
        self.deep_proj = nn.Sequential(
            nn.Conv2d(deep_ch, out_ch, 1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
        # Learnable weight: mô hình tự quyết dùng bao nhiêu từ mỗi scale
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, shallow, deep):
        # Downsample shallow về cùng size với deep
        if shallow.shape[2:] != deep.shape[2:]:
            shallow = nn.functional.adaptive_avg_pool2d(shallow, deep.shape[2:])

        s = self.shallow_proj(shallow)
        d = self.deep_proj(deep)

        # Weighted sum
        alpha = torch.sigmoid(self.alpha)
        return alpha * s + (1 - alpha) * d
        
# ============================================================================
# RViT (Recurrent Vision Transformer)
# ============================================================================
class RViT(nn.Module):
    def __init__(self, shallow_channels, deep_channels, d_model=512, num_patches=1600,
                 n_heads=8, num_encoder_layers=3, dim_feedforward=2048,
                 dropout_rate=0.3, max_seq_length=15):
        super().__init__()
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.fusion = LightFusion(shallow_channels, deep_channels, d_model)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=dim_feedforward,
            dropout=dropout_rate, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)

        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model))
        self.region_q = nn.Parameter(torch.zeros(1, 1, d_model))

        self.embed = nn.Embedding(NUM_CLASSES, d_model)
        self.gru_num_layers = 2
        self.gru = nn.GRU(
            d_model, d_model, num_layers=self.gru_num_layers,
            batch_first=True,
            dropout=dropout_rate if self.gru_num_layers > 1 else 0
        )
        self.attn = MultiheadAttention(
            d_model, num_heads=n_heads, batch_first=True, dropout=dropout_rate
        )
        self.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(2 * d_model, NUM_CLASSES)
        )

    def forward(self, shallow_fmap, deep_fmap, target=None, teach_ratio=0.5, forced_output_length=None):
        b = shallow_fmap.size(0)
        x = self.fusion(shallow_fmap, deep_fmap)
        x = x.flatten(2).permute(0, 2, 1)

        current_num_patches = x.size(1)
        expected_pos_embed_len = current_num_patches + 1

        if self.pos_embed.size(1) != expected_pos_embed_len:
            if self.pos_embed.size(1) > expected_pos_embed_len:
                pos_embed_to_add = self.pos_embed[:, :expected_pos_embed_len, :]
            else:
                raise ValueError(
                    f"RViT pos_embed dim {self.pos_embed.size(1)} < required {expected_pos_embed_len}"
                )
        else:
            pos_embed_to_add = self.pos_embed

        q = self.region_q.expand(b, -1, -1)
        x = torch.cat([q, x], dim=1)
        x = x + pos_embed_to_add

        enc = self.encoder(x)
        region_feat, spatial_feats = enc[:, 0], enc[:, 1:]

        if forced_output_length is not None:
            max_gen_len = forced_output_length
        elif target is not None:
            max_gen_len = target.size(1) - 1
        else:
            max_gen_len = self.max_seq_length - 1

        h = region_feat.unsqueeze(0).repeat(self.gru_num_layers, 1, 1).contiguous()
        current_input_tokens = torch.full((b,), SOS_TOKEN, device=shallow_fmap.device, dtype=torch.long)
        outputs_logits = []

        finished_sequences_tracker = None
        if target is None and forced_output_length is None:
            finished_sequences_tracker = torch.zeros(b, dtype=torch.bool, device=shallow_fmap.device)

        for t in range(max_gen_len):
            emb = self.embed(current_input_tokens).unsqueeze(1)
            g, h = self.gru(emb, h)
            a, _ = self.attn(g, spatial_feats, spatial_feats)
            comb = torch.cat([g.squeeze(1), a.squeeze(1)], dim=-1)
            logits_step = self.fc(comb)
            outputs_logits.append(logits_step)

            if target is not None and random.random() < teach_ratio:
                next_input_candidate = target[:, t + 1]
            else:
                next_input_candidate = logits_step.argmax(-1)

            if finished_sequences_tracker is not None:
                eos_predicted_this_step = (next_input_candidate == EOS_TOKEN)
                finished_sequences_tracker |= eos_predicted_this_step
                current_input_tokens = torch.where(
                    finished_sequences_tracker,
                    torch.tensor(EOS_TOKEN, device=shallow_fmap.device, dtype=torch.long),
                    next_input_candidate
                )
                if finished_sequences_tracker.all():
                    break
            else:
                current_input_tokens = next_input_candidate

        return torch.stack(outputs_logits, dim=1)


# ============================================================================
# FULL MODEL (YOLO_RViT)
# ============================================================================
class YOLO_RViT(nn.Module):
    def __init__(self, yolo_path, shallow_layer=6, deep_layer=13, max_seq_length=15):
        super().__init__()
        self.backbone = YoloBackbone(yolo_path, shallow_layer, deep_layer)

        dummy = torch.randn(1, 3, 640, 640).to(DEVICE)
        with torch.no_grad():
            shallow, deep = self.backbone(dummy)

        self.rvit = RViT(
            shallow_channels=shallow.shape[1],
            deep_channels=deep.shape[1],
            num_patches=deep.shape[2] * deep.shape[3],
            max_seq_length=max_seq_length
        ).to(DEVICE)

    def forward(self, x, target=None, teach_ratio=0.5, forced_output_length=None):
        shallow, deep = self.backbone(x.to(DEVICE))
        return self.rvit(shallow, deep, target, teach_ratio, forced_output_length)

# ============================================================================
# DATASET
# ============================================================================
class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, license_dir, max_seq_length=15, is_train=True):
        self.img_dir = img_dir
        self.license_dir = license_dir
        self.max_seq_length = max_seq_length
        self.img_names = [f for f in os.listdir(self.img_dir) if f.endswith('.jpg')]

        if is_train:
            self.transform = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.RandomApply([
                    transforms.RandomRotation(10),
                    transforms.RandomAffine(degrees=8, translate=(0.03, 0.03), scale=(0.95, 1.05)),
                    transforms.RandomPerspective(distortion_scale=0.05, p=0.3),
                ], p=0.7),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0))], p=0.25),
                transforms.ToTensor(),
                transforms.RandomErasing(p=0.2, scale=(0.01, 0.04), ratio=(0.3, 3.3)),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((640, 640)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])

    def __len__(self):
        return len(self.img_names)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_names[idx])
        img = Image.open(img_path).convert("RGB")
        img_tensor = self.transform(img)

        license_filename = os.path.splitext(self.img_names[idx])[0] + ".txt"
        license_path = os.path.join(self.license_dir, license_filename)

        with open(license_path, 'r', encoding='utf-8') as f:
            license_text = "".join([line.strip() for line in f])

        license_indices = char_to_indices(license_text)
        target = torch.full((self.max_seq_length,), PAD_TOKEN, dtype=torch.long)

        actual_len = min(len(license_indices), self.max_seq_length)
        target[:actual_len] = license_indices[:actual_len]

        return img_tensor, target

    @staticmethod
    def collate_fn(batch):
        images = torch.stack([item[0] for item in batch])
        targets = torch.stack([item[1] for item in batch])
        return images, targets

# ============================================================================
# HYPERPARAMETERS
# ============================================================================
YOLO_MODEL_PATH = '/kaggle/input/models/thiettnnnnnn/t-yolov11s-vnlp/pytorch/default/1/last.pt'

IMG_DIR_TRAIN = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/images/train"
LICENSE_DIR_TRAIN = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/text/train"
IMG_DIR_VAL = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/images/val"
LICENSE_DIR_VAL = "/kaggle/input/datasets/thittnt/rodosal-alpr-dataset/RodoSol-ALPR_dataset/text/val"

MAX_SEQ_LENGTH = 15
BATCH_SIZE = 32
NUM_WORKERS = 4
LEARNING_RATE = 5e-5
MAX_LR_SCHEDULER = 5e-4
WEIGHT_DECAY = 5e-5
NUM_EPOCHS = 100
ACCUM_STEPS = 2
PATIENCE_EARLY_STOP = 7
TEACH_RATIO_START = 0.7
TEACH_RATIO_END = 0.05
LABEL_SMOOTHING = 0.1

# ============================================================================
# SETUP
# ============================================================================
scaler = torch.amp.GradScaler(DEVICE)
autocast_context = lambda: torch.amp.autocast(DEVICE)

train_dataset_full = LicensePlateDataset(
    img_dir=IMG_DIR_TRAIN, license_dir=LICENSE_DIR_TRAIN,
    max_seq_length=MAX_SEQ_LENGTH, is_train=True
)
val_dataset = LicensePlateDataset(
    img_dir=IMG_DIR_VAL, license_dir=LICENSE_DIR_VAL,
    max_seq_length=MAX_SEQ_LENGTH, is_train=False
)

train_dataloader = DataLoader(
    train_dataset_full, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda'), drop_last=True
)
val_dataloader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, collate_fn=LicensePlateDataset.collate_fn,
    pin_memory=(DEVICE == 'cuda')
)

model = YOLO_RViT(
    YOLO_MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Mỗi accumulation cycle = 1 optimizer step
# Số batch cuối cùng cũng trigger 1 step nếu không chia hết
num_batches = len(train_dataloader)
steps_per_epoch = (num_batches + ACCUM_STEPS - 1) // ACCUM_STEPS

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=MAX_LR_SCHEDULER,
    epochs=NUM_EPOCHS,
    steps_per_epoch=steps_per_epoch,
    pct_start=0.2,
    div_factor=(MAX_LR_SCHEDULER / LEARNING_RATE) if MAX_LR_SCHEDULER > LEARNING_RATE else 10.0
)
scheduler_type = "OneCycleLR"

loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_TOKEN, label_smoothing=LABEL_SMOOTHING)

# --- Optional: Load checkpoint ---
# checkpoint = torch.load("...", map_location=DEVICE)
# model.load_state_dict(checkpoint['model_state_dict'])
# optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# ============================================================================
# TRAINING LOOP
# ============================================================================
train_loss_values, val_loss_values = [], []
train_acc_values, val_acc_values, val_acc_sequence_values = [], [], []
epoch_count_list = []
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    epoch_count_list.append(epoch + 1)

    # --- Teach ratio decay ---
    teach_ratio = TEACH_RATIO_START - (TEACH_RATIO_START - TEACH_RATIO_END) * (epoch / max(1, NUM_EPOCHS - 1))

    # ================================================================
    # TRAIN PHASE
    # ================================================================
    model.train()
    train_loss, train_correct, train_total_chars = 0, 0, 0

    pbar_train = tqdm(
        train_dataloader,
        desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [TRAIN] LR: {optimizer.param_groups[0]['lr']:.2e} "
             f"Teach: {teach_ratio:.2f} Scheduler: {scheduler_type}"
    )

    for batch_idx, (imgs, targets) in enumerate(pbar_train):
        imgs = imgs.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        # --- Forward + Loss ---
        with autocast_context():
            outputs = model(imgs, target=targets, teach_ratio=teach_ratio)
            # outputs: [B, seq_len, NUM_CLASSES], targets[:, 1:]: [B, seq_len]
            flat_outputs = outputs.reshape(-1, NUM_CLASSES)
            flat_targets = targets[:, 1:].reshape(-1)
            loss = loss_fn(flat_outputs, flat_targets)
            loss = loss / ACCUM_STEPS

        # --- Backward ---
        scaler.scale(loss).backward()

        # --- Optimizer step (gradient accumulation) ---
        if (batch_idx + 1) % ACCUM_STEPS == 0 or (batch_idx + 1) == num_batches:
            torch.nn.utils.clip_grad_norm_(
                (p for p in model.parameters() if p.requires_grad),
                max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if scheduler_type == "OneCycleLR":
                scheduler.step()

        # --- Train metrics (không cần gradient) ---
        train_loss += loss.item() * ACCUM_STEPS
        with torch.no_grad():
            preds = outputs.argmax(-1)
            true_chars = targets[:, 1:]

            for i in range(imgs.size(0)):
                pred_content, true_content = extract_pred_and_true(
                    preds[i].tolist(), true_chars[i].tolist()
                )
                correct, total = compute_crr(pred_content, true_content)
                train_correct += correct
                train_total_chars += total

            # --- Print examples (batch 0 mỗi epoch) ---
            if batch_idx == 0 and imgs.size(0) > 0:
                print("\n--- Training Batch 0 Examples ---")
                for i in range(min(5, imgs.size(0))):
                    pred_example = index_to_char(preds[i].tolist())
                    true_example = index_to_char(true_chars[i].tolist())
                    print(f"  Pred: '{pred_example}'")
                    print(f"  True: '{true_example}'")
                print("-------------------------------")

        pbar_train.set_postfix(loss=loss.item() * ACCUM_STEPS)

    avg_train_loss = train_loss / num_batches if num_batches > 0 else 0
    avg_train_acc = train_correct / train_total_chars if train_total_chars > 0 else 0
    train_loss_values.append(avg_train_loss)
    train_acc_values.append(avg_train_acc)

    # ================================================================
    # VALIDATION PHASE
    # ================================================================
    model.eval()
    val_loss = 0
    val_correct, val_total_chars = 0, 0
    val_correct_sequences, val_total_sequences = 0, 0
    total_inference_time = 0.0
    num_samples = 0

    pbar_val = tqdm(val_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [VAL]")

    with torch.no_grad():
        for imgs, targets in pbar_val:
            imgs = imgs.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            batch_size = imgs.size(0)
            num_samples += batch_size

            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)

            start_event.record()
            with autocast_context():
                outputs = model(imgs, target=None, teach_ratio=0.0)
            end_event.record()
            torch.cuda.synchronize()

            inference_time_ms = start_event.elapsed_time(end_event)
            total_inference_time += inference_time_ms
            
            with autocast_context():
                out_seq_len = outputs.size(1)
                tgt_content_len = targets.size(1) - 1  # Bỏ SOS ở đầu target

                # Lấy phần ngắn hơn giữa output và target
                min_len = min(out_seq_len, tgt_content_len)
                outputs_for_loss = outputs[:, :min_len, :]
                targets_for_loss = targets[:, 1:min_len + 1]  # Bỏ SOS, lấy min_len tokens

                flat_outputs_val = outputs_for_loss.reshape(-1, NUM_CLASSES)
                flat_targets_val = targets_for_loss.reshape(-1)
                loss = loss_fn(flat_outputs_val, flat_targets_val)

            val_loss += loss.item()

            preds_val = outputs.argmax(-1) 
            true_chars_val = targets[:, 1:]

            for i in range(batch_size):
                pred_content, true_content = extract_pred_and_true(
                    preds_val[i].tolist(), true_chars_val[i].tolist()
                )

                # CRR
                correct, total = compute_crr(pred_content, true_content)
                val_correct += correct
                val_total_chars += total

                # E2E exact match (toàn bộ biển số phải đúng hoàn toàn)
                if pred_content == true_content:
                    val_correct_sequences += 1
                val_total_sequences += 1

            pbar_val.set_postfix(loss=loss.item())

    # ================================================================
    # EPOCH SUMMARY
    # ================================================================
    avg_val_loss = val_loss / len(val_dataloader) if len(val_dataloader) > 0 else 0
    avg_val_acc = val_correct / val_total_chars if val_total_chars > 0 else 0
    avg_val_sequence_accuracy = val_correct_sequences / val_total_sequences if val_total_sequences > 0 else 0.0

    avg_inference_time = total_inference_time / num_samples if num_samples > 0 else 0  # ms/sample
    fps = 1000.0 / avg_inference_time if avg_inference_time > 0 else 0  # FPS

    val_loss_values.append(avg_val_loss)
    val_acc_values.append(avg_val_acc)
    val_acc_sequence_values.append(avg_val_sequence_accuracy)

    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e} "
          f"| Teach: {teach_ratio:.2f} | Scheduler: {scheduler_type}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Train CRR: {avg_train_acc:.4f}")
    print(f"  Val Loss:   {avg_val_loss:.4f} | Val CRR:   {avg_val_acc:.4f}")
    print(f"  Val E2E RR: {avg_val_sequence_accuracy:.4f}")
    print(f"  Inference:  {avg_inference_time:.2f} ms/img | FPS: {fps:.1f}")
    print("-" * 70)

    # --- Save best model ---
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        print(f"*** New best CRR: {best_val_acc:.4f}. Saving best_model.pth ***")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_loss': avg_val_loss,
            'val_crr': avg_val_acc,
            'val_e2e_rr': avg_val_sequence_accuracy,
            'avg_inference_time_ms': avg_inference_time,
            'fps': fps
        }, "best_yolo_rvit_model.pth")

# ============================================================================
# SAVE FINAL MODEL
# ============================================================================
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'scheduler_state_dict': scheduler.state_dict(),
    'train_loss_history': train_loss_values,
    'val_loss_history': val_loss_values,
    'train_acc_history': train_acc_values,
    'val_acc_history': val_acc_values,
    'val_acc_sequence_history': val_acc_sequence_values,
    'avg_inference_time_ms': avg_inference_time,
    'fps': fps
}, "final_yolo_rvit_model.pth")

print("\nTraining completed!")
if val_acc_values:
    print(f"Final Val CRR:    {val_acc_values[-1]:.4f}")
if val_acc_sequence_values:
    print(f"Final Val E2E RR: {val_acc_sequence_values[-1]:.4f}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


/tmp/ipykernel_24/2281152128.py:177: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_encoder_layers)
Epoch 1/100 [TRAIN] LR: 5.00e-05 Teach: 0.70 Scheduler: OneCycleLR:   0%|          | 1/250 [00:15<1:03:07, 15.21s/it, loss=3.69]


--- Training Batch 0 Examples ---
  Pred: 'WXTXXXXXG'
  True: 'QFR0H07'
  Pred: 'WPPGXRGGNGKG'
  True: 'QRK5D17'
  Pred: 'TTXM9X'
  True: 'MTL0617'
  Pred: 'LXBXDXGGQGL'
  True: 'QRE3D65'
  Pred: 'H9XX992XGXXGGG'
  True: 'ODE4307'
-------------------------------


Epoch 1/100 [TRAIN] LR: 5.00e-05 Teach: 0.70 Scheduler: OneCycleLR: 100%|██████████| 250/250 [05:32<00:00,  1.33s/it, loss=2.9]
Epoch 1/100 [VAL]: 100%|██████████| 125/125 [01:11<00:00,  1.74it/s, loss=3.12]



Epoch 1/100 | LR: 5.28e-05 | Teach: 0.70 | Scheduler: OneCycleLR
  Train Loss: 3.0883 | Train CRR: 0.0744
  Val Loss:   3.0241 | Val CRR:   0.1050
  Val E2E RR: 0.0000
  Inference:  11.38 ms/img | FPS: 87.9
----------------------------------------------------------------------
*** New best CRR: 0.1050. Saving best_model.pth ***


...


Epoch 99/100 | LR: 1.95e-07 | Teach: 0.06 | Scheduler: OneCycleLR
  Train Loss: 0.9850 | Train CRR: 0.8937
  Val Loss:   0.8558 | Val CRR:   0.9394
  Val E2E RR: 0.7505
  Inference:  11.90 ms/img | FPS: 84.0
----------------------------------------------------------------------


Epoch 100/100 [TRAIN] LR: 1.95e-07 Teach: 0.05 Scheduler: OneCycleLR:   0%|          | 1/250 [00:05<23:42,  5.71s/it, loss=1.02]


--- Training Batch 0 Examples ---
  Pred: 'PPU0C57'
  True: 'PPU0C57'
  Pred: 'MQG5G29'
  True: 'MQO5G29'
  Pred: 'OVG1C78'
  True: 'OYD3C78'
  Pred: 'MRF3399'
  True: 'MRF3399'
  Pred: 'ODR1855'
  True: 'ODR1856'
-------------------------------


Epoch 100/100 [TRAIN] LR: 1.95e-07 Teach: 0.05 Scheduler: OneCycleLR: 100%|██████████| 250/250 [05:19<00:00,  1.28s/it, loss=0.979]
Epoch 100/100 [VAL]: 100%|██████████| 125/125 [01:00<00:00,  2.06it/s, loss=0.827]



Epoch 100/100 | LR: 5.01e-09 | Teach: 0.05 | Scheduler: OneCycleLR
  Train Loss: 0.9849 | Train CRR: 0.8932
  Val Loss:   0.8559 | Val CRR:   0.9398
  Val E2E RR: 0.7512
  Inference:  11.96 ms/img | FPS: 83.6
----------------------------------------------------------------------

Training completed!
Final Val CRR:    0.9398
Final Val E2E RR: 0.7512
